<a href="https://colab.research.google.com/github/ALcitya/Penerjemah_SIBI/blob/main/deteksi_isyarat_sibi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

untuk mengerjakan skripsi penerjemah bahasa isyarat sibi mengunakan cnn dan lstm

# import library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import cv2
import os
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, TimeDistributed, Flatten, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## memanggil folder data

In [3]:
# Tentukan path ke folder 'raw' Anda. Sesuaikan jika folder 'raw' berada di subfolder lain.
raw_folder_path = '/content/drive/MyDrive/raw'

# Periksa apakah folder tersebut ada
if os.path.exists(raw_folder_path):
    print(f"Folder 'raw' ditemukan di: {raw_folder_path}")
    # Anda bisa mulai memproses file di dalam folder 'raw'
    # Misalnya, tampilkan daftar isinya:
    print("Isi folder 'raw':")
    for item in os.listdir(raw_folder_path):
        print(item)
else:
    print(f"Folder 'raw' tidak ditemukan di: {raw_folder_path}")
    print("Pastikan folder 'raw' ada di Google Drive Anda dan path yang diberikan sudah benar.")

Folder 'raw' ditemukan di: /content/drive/MyDrive/raw
Isi folder 'raw':
tinggal
toko
waktu
tutup
tunggu
uang
teman
teh
tidur
telepon
tanya
tahu
tas
suka
sulit
siapa
senang
selesai
sore
selamat
sekolah
sedih
sekarang
salah
rumah
saya
pasar
perlu
pergi
partikel-lah
partikel-kah
partike-pun
pagi
orang
nasi
mulai
nanti
mudah
minum
mereka
motor
mengapa
mana
meja
mau
malam
lihat
makan
lupa
kursi
kita
kerja
keluarga
jalan-jalan
kendara
guru
guna
ini
jalan
jawab
itu
dengar
kapan
gedung
kamar
ingat
bicara
duduk
kami
cari
kamu
buka
beli
dosen
hari
benar
datang
bawa
bantu
bisa
berapa
bagaimana
dia
desa
buku
awalan-ter
balai
baca
awalan-se
awalan-ke
awalan-pe
apa
awalan-me
awalan-di
awalan-ber
akhiran-man
angka
akhiran-kan
akhiran-wan
akhiran-nya
akhiran-ti
akhiran-an
akhiran-i
akhiran-wati


# Preprocessing

In [4]:
root_raw_dir = raw_folder_path
output_root = 'data/processed'

num_frames_to_save = 20
target_size = (128,128)

os.makedirs(os.path.join(output_root, 'rgb'), exist_ok=True)
os.makedirs(os.path.join(output_root, 'grayscale'), exist_ok=True)

for label in os.listdir(root_raw_dir):

    word_path = os.path.join(root_raw_dir, label)

    if not os.path.isdir(word_path):
        continue

    print(f"Processing label: {label}")

    rgb_label_dir = os.path.join(output_root, 'rgb', label)
    gray_label_dir = os.path.join(output_root, 'grayscale', label)

    os.makedirs(rgb_label_dir, exist_ok=True)
    os.makedirs(gray_label_dir, exist_ok=True)

    for video_name in os.listdir(word_path):

        if not video_name.lower().endswith('.webm'):
            continue

        video_full_path = os.path.join(word_path, video_name)
        video_base_name = os.path.splitext(video_name)[0]

        cap = cv2.VideoCapture(video_full_path)

        if not cap.isOpened():
            print(f"Gagal membuka video: {video_name}")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            print(f"Video rusak: {video_name}")
            cap.release()
            continue

        step = max(1, total_frames // num_frames_to_save)

        for i in range(num_frames_to_save):

            frame_id = i * step
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)

            ret, frame = cap.read()
            if not ret:
                break

            frame_res = cv2.resize(frame, target_size)
            # crop bagian atas
            h, w = frame_res.shape[:2]
            crop_top = int(h * 0.25)
            frame_crop = frame_res[crop_top:h, 0:w]

            # resize lagi agar tetap 128x128
            frame_crop = cv2.resize(frame_crop, target_size)

            rgb_name = f"{video_base_name}_f{i:02d}.jpg"
            cv2.imwrite(os.path.join(rgb_label_dir, rgb_name), frame_crop)

            gray = cv2.cvtColor(frame_crop, cv2.COLOR_BGR2GRAY)
            gray_name = f"{video_base_name}_f{i:02d}.jpg"
            cv2.imwrite(os.path.join(gray_label_dir, gray_name), gray)

        cap.release()

        print(f"   selesai: {video_name}")

print("\nSemua video selesai diproses.")

Processing label: tinggal
   selesai: Tinggal.webm
Processing label: toko
   selesai: toko.webm
Processing label: waktu
   selesai: Waktu.webm
Processing label: tutup
   selesai: Tutup.webm
Processing label: tunggu
   selesai: Tunggu.webm
Processing label: uang
   selesai: Uang.webm
Processing label: teman
   selesai: Teman.webm
Processing label: teh
   selesai: Teh.webm
Processing label: tidur
   selesai: Tidur.webm
Processing label: telepon
   selesai: Telepon.webm
Processing label: tanya
   selesai: Tanya.webm
Processing label: tahu
   selesai: Tahu.webm
Processing label: tas
   selesai: Tas.webm
Processing label: suka
   selesai: Suka.webm
Processing label: sulit
   selesai: Sulit.webm
Processing label: siapa
   selesai: Siapa.webm
Processing label: senang
   selesai: Senang.webm
Processing label: selesai
   selesai: Selesai.webm
Processing label: sore
   selesai: Sore.webm
Processing label: selamat
   selesai: Selamat.webm
Processing label: sekolah
   selesai: Sekolah.webm
Process

# Augmentasi
untuk memperkaya data pelatihan

## Augmentasi RGB

In [5]:
INPUT_DIR = "data/processed/rgb"
OUTPUT_DIR = "data/augmented/rgb"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for label in os.listdir(INPUT_DIR):

    label_path = os.path.join(INPUT_DIR, label)
    output_label_path = os.path.join(OUTPUT_DIR, label)

    os.makedirs(output_label_path, exist_ok=True)

    print("Processing label:", label)

    for img_name in os.listdir(label_path):

        img_path = os.path.join(label_path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        base_name = os.path.splitext(img_name)[0]

        # ORIGINAL
        cv2.imwrite(os.path.join(output_label_path, base_name + "_orig.jpg"), img)

        # FLIP
        flip = cv2.flip(img, 1)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_flip.jpg"), flip)

        # ROTATE
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), 10, 1)
        rotate = cv2.warpAffine(img, M, (w, h))
        cv2.imwrite(os.path.join(output_label_path, base_name + "_rot.jpg"), rotate)

        # BRIGHTNESS
        bright = cv2.convertScaleAbs(img, alpha=1.2, beta=20)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_bright.jpg"), bright)

        # GAUSSIAN NOISE
        noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
        noisy = cv2.add(img, noise)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_noise.jpg"), noisy)

        # ZOOM
        zoom_factor = 1.2
        zoom = cv2.resize(img, None, fx=zoom_factor, fy=zoom_factor)
        zh, zw = zoom.shape[:2]
        zoom_crop = zoom[(zh-h)//2:(zh-h)//2+h, (zw-w)//2:(zw-w)//2+w]
        cv2.imwrite(os.path.join(output_label_path, base_name + "_zoom.jpg"), zoom_crop)

        # GAUSSIAN BLUR
        blur = cv2.GaussianBlur(img, (5,5), 0)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_blur.jpg"), blur)

print("Augmentasi RGB selesai!")

Processing label: datang
Processing label: partike-pun
Processing label: tunggu
Processing label: akhiran-ti
Processing label: guna
Processing label: motor
Processing label: salah
Processing label: waktu
Processing label: duduk
Processing label: suka
Processing label: jalan-jalan
Processing label: kami
Processing label: sore
Processing label: kamar
Processing label: mudah
Processing label: ini
Processing label: benar
Processing label: selamat
Processing label: lihat
Processing label: angka
Processing label: sekolah
Processing label: pagi
Processing label: telepon
Processing label: senang
Processing label: teman
Processing label: nasi
Processing label: awalan-ke
Processing label: mulai
Processing label: dengar
Processing label: awalan-di
Processing label: itu
Processing label: buka
Processing label: beli
Processing label: dia
Processing label: rumah
Processing label: hari
Processing label: kita
Processing label: jawab
Processing label: mereka
Processing label: tutup
Processing label: or

## Augmentasion Grayscale

In [3]:
INPUT_DIR = "data/processed/grayscale"
OUTPUT_DIR = "data/augmented/grayscale"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for label in os.listdir(INPUT_DIR):

    label_path = os.path.join(INPUT_DIR, label)
    output_label_path = os.path.join(OUTPUT_DIR, label)

    os.makedirs(output_label_path, exist_ok=True)

    print("Processing label:", label)

    for img_name in os.listdir(label_path):

        img_path = os.path.join(label_path, img_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        base_name = os.path.splitext(img_name)[0]

        # ========================
        # ORIGINAL
        # ========================
        cv2.imwrite(os.path.join(output_label_path, base_name + "_orig.jpg"), img)

        # ========================
        # FLIP
        # ========================
        flip = cv2.flip(img, 1)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_flip.jpg"), flip)

        # ========================
        # ROTATE
        # ========================
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w//2, h//2), 10, 1)
        rotate = cv2.warpAffine(img, M, (w, h))
        cv2.imwrite(os.path.join(output_label_path, base_name + "_rot.jpg"), rotate)

        # ========================
        # BRIGHTNESS
        # ========================
        bright = cv2.convertScaleAbs(img, alpha=1.2, beta=20)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_bright.jpg"), bright)

        # ========================
        # GAUSSIAN NOISE (FIXED)
        # ========================
        noise = np.random.normal(0, 10, img.shape)
        noisy = img.astype(np.float32) + noise
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_noise.jpg"), noisy)

        # ========================
        # ZOOM
        # ========================
        zoom_factor = 1.2
        zoom = cv2.resize(img, None, fx=zoom_factor, fy=zoom_factor)
        zh, zw = zoom.shape
        zoom_crop = zoom[(zh-h)//2:(zh-h)//2+h, (zw-w)//2:(zw-w)//2+w]
        cv2.imwrite(os.path.join(output_label_path, base_name + "_zoom.jpg"), zoom_crop)

        # ========================
        # GAUSSIAN BLUR
        # ========================
        blur = cv2.GaussianBlur(img, (5,5), 0)
        cv2.imwrite(os.path.join(output_label_path, base_name + "_blur.jpg"), blur)

print("Augmentasi grayscale selesai!")

Processing label: datang
Processing label: partike-pun
Processing label: tunggu
Processing label: akhiran-ti
Processing label: guna
Processing label: motor
Processing label: salah
Processing label: waktu
Processing label: duduk
Processing label: suka
Processing label: jalan-jalan
Processing label: kami
Processing label: sore
Processing label: kamar
Processing label: mudah
Processing label: ini
Processing label: benar
Processing label: selamat
Processing label: lihat
Processing label: angka
Processing label: sekolah
Processing label: pagi
Processing label: telepon
Processing label: senang
Processing label: teman
Processing label: nasi
Processing label: awalan-ke
Processing label: mulai
Processing label: dengar
Processing label: awalan-di
Processing label: itu
Processing label: buka
Processing label: beli
Processing label: dia
Processing label: rumah
Processing label: hari
Processing label: kita
Processing label: jawab
Processing label: mereka
Processing label: tutup
Processing label: or

# Preparation
menyiapkan data sebelum ke pelatihan

## Preparation RGB

In [7]:
INPUT_DIR = "data/augmented/rgb"
OUTPUT_X = "data/prep/rgb/X_rgb.npy"
OUTPUT_Y = "data/prep/rgb/Y_rgb.npy"

NUM_FRAMES = 20

def prep_rgb():

    X, Y = [], []

    labels = sorted(os.listdir(INPUT_DIR))
    label_map = {label:i for i,label in enumerate(labels)}

    # Ensure output directory exists
    os.makedirs(os.path.dirname(OUTPUT_X), exist_ok=True)

    for label in labels:

        label_path = os.path.join(INPUT_DIR,label)

        if not os.path.isdir(label_path):
            continue

        print("Processing:",label)

        all_images = sorted(os.listdir(label_path))

        for i in range(0,len(all_images),NUM_FRAMES):

            frame_batch = all_images[i:i+NUM_FRAMES]

            if len(frame_batch) < NUM_FRAMES:
                continue

            video_frames = []

            for img_name in frame_batch:

                path = os.path.join(label_path,img_name)

                img = cv2.imread(path)

                img = img.astype("float32") / 255.0

                video_frames.append(img)

            X.append(video_frames)
            Y.append(label_map[label])

    X = np.array(X)
    Y = np.array(Y)

    np.save(OUTPUT_X,X)
    np.save(OUTPUT_Y,Y)

    print("Selesai")
    print("Shape X:",X.shape)
    print("Shape Y:",Y.shape)

if __name__ == "__main__":
    prep_rgb()

Processing: akhiran-an
Processing: akhiran-i
Processing: akhiran-kan
Processing: akhiran-man
Processing: akhiran-nya
Processing: akhiran-ti
Processing: akhiran-wan
Processing: akhiran-wati
Processing: angka
Processing: apa
Processing: awalan-ber
Processing: awalan-di
Processing: awalan-ke
Processing: awalan-me
Processing: awalan-pe
Processing: awalan-se
Processing: awalan-ter
Processing: baca
Processing: bagaimana
Processing: balai
Processing: bantu
Processing: bawa
Processing: beli
Processing: benar
Processing: berapa
Processing: bicara
Processing: bisa
Processing: buka
Processing: buku
Processing: cari
Processing: datang
Processing: dengar
Processing: desa
Processing: dia
Processing: dosen
Processing: duduk
Processing: gedung
Processing: guna
Processing: guru
Processing: hari
Processing: ingat
Processing: ini
Processing: itu
Processing: jalan
Processing: jalan-jalan
Processing: jawab
Processing: kamar
Processing: kami
Processing: kamu
Processing: kapan
Processing: keluarga
Processing

## Preparation Graycscale

In [5]:
INPUT_DIR = 'data/augmented/grayscale'
OUTPUT_X = 'data/prep/grayscale/X_gray.npy'
OUTPUT_Y = 'data/prep/grayscale/Y_gray.npy'

NUM_FRAMES = 20
IMG_SIZE = 128

os.makedirs(os.path.dirname(OUTPUT_X), exist_ok=True)

def prep_gray():

    X, Y = [], []

    labels = sorted(os.listdir(INPUT_DIR))
    label_map = {label: i for i, label in enumerate(labels)}

    for label in labels:

        label_path = os.path.join(INPUT_DIR, label)

        if not os.path.isdir(label_path):
            continue

        print(f"Processing label: {label}")

        all_images = sorted([
            f for f in os.listdir(label_path)
            if f.endswith(".png") or f.endswith(".jpg")
        ])

        for i in range(0, len(all_images), NUM_FRAMES):

            frame_batch = all_images[i:i+NUM_FRAMES]

            if len(frame_batch) < NUM_FRAMES:
                continue

            video_frames = []

            for img_name in frame_batch:

                path = os.path.join(label_path, img_name)

                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                img = img.astype('float32') / 255.0

                video_frames.append(img)

            X.append(video_frames)
            Y.append(label_map[label])

    X = np.array(X)
    Y = np.array(Y)

    X = np.expand_dims(X, axis=-1)

    np.save(OUTPUT_X, X)
    np.save(OUTPUT_Y, Y)

    print("Selesai!")
    print("Shape X:", X.shape)
    print("Shape Y:", Y.shape)

if __name__ == "__main__":
    prep_gray()

Processing label: akhiran-an
Processing label: akhiran-i
Processing label: akhiran-kan
Processing label: akhiran-man
Processing label: akhiran-nya
Processing label: akhiran-ti
Processing label: akhiran-wan
Processing label: akhiran-wati
Processing label: angka
Processing label: apa
Processing label: awalan-ber
Processing label: awalan-di
Processing label: awalan-ke
Processing label: awalan-me
Processing label: awalan-pe
Processing label: awalan-se
Processing label: awalan-ter
Processing label: baca
Processing label: bagaimana
Processing label: balai
Processing label: bantu
Processing label: bawa
Processing label: beli
Processing label: benar
Processing label: berapa
Processing label: bicara
Processing label: bisa
Processing label: buka
Processing label: buku
Processing label: cari
Processing label: datang
Processing label: dengar
Processing label: desa
Processing label: dia
Processing label: dosen
Processing label: duduk
Processing label: gedung
Processing label: guna
Processing label:

# Training model

In [2]:
# 1. Load data
X = np.load('data/prep/rgb/X_rgb.npy')
Y = np.load('data/prep/rgb/Y_rgb.npy')

# Normalisasi
X = X.astype('float32') / 255.0

# Jika Y masih berupa label (bukan one-hot)
if len(Y.shape) > 1:
    labels = np.argmax(Y, axis=1)
else:
    labels = Y

# Distribusi kelas
unique, counts = np.unique(labels, return_counts=True)

# Jumlah kelas
num_classes = len(unique)

# Split data
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# One-hot encoding setelah split
Y_train = tf.keras.utils.to_categorical(Y_train, num_classes)
Y_test = tf.keras.utils.to_categorical(Y_test, num_classes)

print("Shape input untuk model:", X.shape[1:])

# 2. Bangun model CNN-LSTM
model = Sequential([

    TimeDistributed(
        Conv2D(32, (3,3), activation='relu'),
        input_shape=X.shape[1:]
    ),
    TimeDistributed(MaxPooling2D((2,2))),

    TimeDistributed(Conv2D(64, (3,3), activation='relu')),
    TimeDistributed(MaxPooling2D((2,2))),

    TimeDistributed(Flatten()),

    # LSTM untuk sequence frame
    LSTM(64, return_sequences=False),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

# 3. Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 4. Training
history = model.fit(
    X_train,
    Y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.1,
    verbose=1
)

# 5. Evaluasi
test_loss, test_acc = model.evaluate(X_test, Y_test)
print(f"\nAkurasi Test: {test_acc*100:.4f}%")

# 6. Plot hasil training
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.show()

# 7. Simpan model
os.makedirs('models', exist_ok=True)
model.save('models/sibi_model_rgb.keras')

Shape input untuk model: (20, 128, 128, 3)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 126, 126,   │           896 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 63, 63, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 61, 61, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 30, 30, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 20, 57600)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │    14,762,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 104)            │         6,760 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,792,552 (56.43 MB)

 Trainable params: 14,792,552 (56.43 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 27s 275ms/step - accuracy: 0.0096 - loss: 4.6490 - val_accuracy: 0.0169 - val_loss: 4.6490
Epoch 2/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 235ms/step - accuracy: 0.0115 - loss: 4.6457 - val_accuracy: 0.0000e+00 - val_loss: 4.6526
Epoch 3/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 236ms/step - accuracy: 0.0057 - loss: 4.6451 - val_accuracy: 0.0000e+00 - val_loss: 4.6546
Epoch 4/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 236ms/step - accuracy: 0.0057 - loss: 4.6444 - val_accuracy: 0.0000e+00 - val_loss: 4.6581
Epoch 5/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 21s 239ms/step - accuracy: 0.0057 - loss: 4.6443 - val_accuracy: 0.0000e+00 - val_loss: 4.6600
Epoch 6/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 243ms/step - accuracy: 0.0115 - loss: 4.6435 - val_accuracy: 0.0000e+00 - val_loss: 4.6630
Epoch 7/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 244ms/step - accuracy: 0.0134 - loss: 4.6430 - val_accuracy: 0.0000e+00 - val_loss: 4.6677
Epoch 8/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 241ms/step - accuracy: 0.0038 -

KeyboardInterrupt: 